# Project 4 quick start

***

## Basic usage

This notebook shows a basic usage of the `deepracer_gym` package along with some utility functions in `src.utils`.

Make sure that you have completed the setup from the [`SETUP.md`](SETUP.md) file and are using the proper python environment with this notebook.

### Manage the simulator service(s)

Instantiating the gym environment `gym.make('deepracer-v0')` spins up a DeepRacer simulation service. The `deepracer_gym` package provides some helper functions to manage these services:

In [ ]:
import deepracer_gym

deepracer_gym.running()         # list running services
deepracer_gym.shutdown_all()    # shutdown all services

The number of simulators can be capped via the environment variable `DEEPRACER_MAX_ENVS` (default 4). If you want to completely clean your environment (shutdown all idle/stray/cached containers), run the following shell comamnd:

```bash
python -m deepracer_gym.clean
```

**Note** the environment takes its action space and sensors from an `agent_config` dict, its world and objects from a `track_config` dict, and its reward from a `reward_function` — all optional arguments to `gym.make` (packaged defaults are used if you omit them). There are no config files to edit: customize by passing dicts / a function, as shown below. See the `deepracer_gym` README (under `packages/`) for the full config schema.

### Interact with simulation via `deepracer_gym`

We can interact with the simulation service using the familiar `gymnasium` API via the `deepracer-v0` environment provided by the `deepracer_gym` package (under `deepracer/client`).

Simply import `deepracer_gym` before using the `deepracer-v0` environment with `gymnasium` as usual.

In [ ]:
import gymnasium as gym
import deepracer_gym

# you can specify simulation parameters via the following 
# (see the deepracer/client/README.md for the full schema):
#   agent_config = {
#       'action_space': [...],
#       'sensor': [...],
#       'action_space_type': 'discrete'
#   }
#   track_config = {
#       'WORLD_NAME': 'reInvent2019_wide',
#       'NUMBER_OF_OBSTACLES': '0',
#       'NUMBER_OF_BOT_CARS': '0',
#       ...
#   }
#   reward_function = lambda params: return 1.0
env = gym.make('deepracer-v0')

observation, info = env.reset()

observation, reward, terminated, truncated, info = env.step(
    env.action_space.sample()
)

# stop and remove simulator service
env.unwrapped.close(
    keep_warm=True      # reuse the service for a future env.make() call
)

In [ ]:
# see the output dimensions of the observations
{
    k: v.shape for k, v in observation.items()
}

***

## Utility functions and features

### Flattened environment

Notice that the `observation` variable above is a dictionary (keys are sensor names, values are measurements). Such a data-structure is a bit more difficult to handle than simple vectors, especially for batching purposes. Therefore, we suggest that you use the provided `src.utils.make_environmrnt` function instead. It 'flattens' the observation space into a single vector space using the `gymnasium.wrappers.FlattenObservation` class.

Additionally, it also wraps the environment with a `gymnasium.wrappers.RecordEpisodeStatistics` class, which can be very convenient for calculating things like episode langth and returns.

Please get familiar with all of these functions/ classes before attempting the project.

In [ ]:
from src.utils import make_environment

env = make_environment(         # just replace gym.make
    'deepracer-v0'
)

observation, info = env.reset()

observation, reward, terminated, truncated, info = env.step(
    env.action_space.sample()
)

env.unwrapped.close(keep_warm=True)

In [ ]:
# see the output dimensions of the observations
observation.shape

***

### Observation encoder

We can un-flatten, pre-process and encode the flattened observations above using an **example/default** encoder implementation provided as `src.transforms.EncodeObservation`. An example usage is given below, feel free to change/ modify this implementation and its components<a name="cite_ref-1"></a>[<sup>[1]</sup>](#cite_note-1) according to your needs.

<a name="cite_note-1"></a>1. [^](#cite_ref-1) Make sure to appropriately pre-process your observations under `src.transforms.PreprocessLiDAR` and `src.transforms.PreprocessCamera`!

In [ ]:
from src.run import tensor
from src.utils import device

from src.transforms import EncodeObservation

DEVICE = device()
# encode the observations
encoder = EncodeObservation().to(DEVICE)
encoded_observation = encoder(
    tensor(observation)
)

# see the output dimensions of the encoded observations
encoded_observation.shape

### Visualize agent policy

So long as your agent implements a `get_action` method as in `src.agents.py`, you can use our provided `src.utils.demo` function to visualize the policy of the agent in the form of a MP4 video saved under `demos/`.

In [ ]:
from src.utils import demo
from src.agents import RandomAgent

agent = RandomAgent(environment=env)
demo(agent.eval())

### Evaluate on multiple tracks

You can use the provided `src.utils.evaluate` function to evaluate your agent on all three project tracks (for the race type implied by your `track_config`). The results are both returned and saved (overwritten) under `evaluations/`.

This provisions a fresh simulator for each of the three tracks, so it may take a few minutes (roughly ~5 minutes for an untrained agent, longer for a trained one).

In [ ]:
from src.utils import evaluate

metrics = evaluate(
    agent.eval()
)

### Plotting evaluation metrics

You can plot the returned evaluation metrics dictionary using the provided `src.utils.plot_metrics` functions. The results are also saved under `plots/`.

Note however that you do not necessarily have to stick to this exact visualization/plot, and may make adjustments as you see fit.

In [ ]:
from src.utils import plot_metrics

plot_metrics(
    metrics, title='usage'
)

***

## Training and logging

Please use the structure in `src.run.py` to design your training and logging loops.

In [ ]:
from src.run import run

run()

To view the training logs, you can use `tensorboard --logdir runs`. On PACE ICE, run the following command in your terminal:

```bash
PORT=$(python -c "import socket; s=socket.socket(); s.bind(('',0)); print(s.getsockname()[1]); s.close()")
NODE=$(hostname)

tensorboard \
  --logdir runs \
  --host 0.0.0.0 \
  --port "$PORT" \
  --path_prefix "/node/$NODE/$PORT" &

echo
echo "TensorBoard live at:"
echo "https://ondemand-ice.pace.gatech.edu/node/$NODE/$PORT/"
```

***